# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [ ]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [ ]:
q1 = q('''
SELECT t.title, a.name, a.country
FROM tracks t
JOIN artists a
ON t.artist_id = a.artist_id;
''')

q1

,title,name,country
0,Skyline,Nova Waves,US
1,Undertow,Nova Waves,US
2,Foothills,The Blue Ridge,US
3,Aurora,Kestrel,UK
4,Nightfall,Kestrel,UK
5,Sol,Marisol,ES
6,Coastline,The Blue Ridge,US
7,Ridgeline,The Blue Ridge,US
8,Untitled Demo,Kestrel,UK


The 'tracks' table contains the title and the 'artist' table contains the name and country. I joined the two tables on 'artist_id' because that is there common factor to display the title, name, and country for each song.

### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [ ]:
q('''
SELECT title, genre, seconds
FROM tracks t;
''')



,title,genre,seconds
0,Skyline,Pop,201
1,Undertow,Pop,240
2,Foothills,Folk,185
3,Aurora,Electronic,300
4,Nightfall,Electronic,275
5,Sol,Latin,210
6,Coastline,Folk,199
7,Ridgeline,Folk,225
8,Untitled Demo,None,150


I first wanted to check that I was searching for the correct column titles.

In [ ]:
q2 = q('''
SELECT genre, AVG(seconds) AS avg_seconds
FROM  tracks t
WHERE genre IS NOT NULL
GROUP BY genre
ORDER BY avg_seconds DESC
LIMIT 1;
''')

q2

,genre,avg_seconds
0,Electronic,287.5


I found all of the columns needed in the tracks table, and took the average of each genre's time sorting in descending order, with a limit of 1, to display the genre with the longer average track time.

### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [ ]:
q3 = q('''
SELECT user,
COUNT(*) AS total_plays,
COUNT(DISTINCT track_id) AS distinct_tracks_plays
FROM plays
GROUP BY user;
''')

q3

,user,total_plays,distinct_tracks_plays
0,ava,4,4
1,ben,3,3
2,cara,2,2
3,dan,2,2


Using the plays table, I counted every row to get total plays and distinct trakcs to prevent overlap, displaying distinct track plays. These both happened to be the same.

### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [ ]:
q4 = q('''
SELECT t.track_id, t.title
FROM tracks t
LEFT JOIN plays p
ON t.track_id = p. track_id
WHERE p.play_id IS NULL;
''')

q4

,track_id,title
0,17,Ridgeline
1,18,Untitled Demo


I used a left join to find all of the tracks that id has a corresponding 'NULL' value in the 'plays' table. This display 2 rows, as expected.

### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [ ]:
q5 = q('''
SELECT a.name AS artist,
SUM(t.seconds) AS total_seconds,
ROUND(SUM(t.seconds) / 60, 1) AS total_minutes
FROM artists a
JOIN tracks t
ON a.artist_id = t.artist_id
JOIN plays p
ON t.track_id = p.track_id
GROUP BY a.name
ORDER BY total_seconds DESC;
''')

q5

,artist,total_seconds,total_minutes
0,Kestrel,1175,19.0
1,Nova Waves,843,14.0
2,The Blue Ridge,384,6.0
3,Marisol,210,3.0


I joined all three table to sum the time of plays for each track (both in seconds and minutes) in descending order to display the artists in order of who had the most play time.

### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [ ]:
q6 = q('''
SELECT t.track_id, t.title, t.genre
FROM tracks t
WHERE genre is NULL
''')

q6

,track_id,title,genre
0,18,Untitled Demo,None


WHERE genre != 'Pop' would have displayed the rows it knows do not fall under the 'Pop' genre. This goes back to what was talked about in lecture last week; NULL in SQL doesnt mean 'nothing' it means 'unknown'.

### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [ ]:
q7 = q('''
SELECT p.played_on,
COUNT(*) AS total_plays,
COUNT(DISTINCT user) AS distinct_users
FROM plays p
GROUP BY played_on
ORDER BY played_on ASC;
''')

q7

,played_on,total_plays,distinct_users
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


Use the 'plays' table, I used COUNT(*) to count all plays and COUNT(DISTINCT user) to count the plays by each user active in a given day. I Ordered by ascending to show the earilest date first.

### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [ ]:
 assert len(q1) == 9, 'Q1 should return one row per track'
 assert len(q4) == 2, 'Q4: two tracks have never been played'
#assert q3['plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


I had to go back at title each cell 'qn' for the validation to work.

### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

I found q4 to give me the most trouble because I initally could not the remember the difference between 'JOIN' and 'LEFT_JOIN'. Additonally, I forget that '=NULL' and 'is NULL' are not the same thing. I had to research to remember "LEFT_JOIN' is used to join two tables and maintain rows of the left table that do not overlap. By not understanding how NULL works in SQL, I was pretty much claiming the two 'NULL' values were equal which was confusing the notebook when in reality they are unknown.

_your answer here_